# Instalar dependências

In [2]:
!pip install -q transformers accelerate peft bitsandbytes datasets trl

^C


# Login no Hugging Face (necessário para LLaMA)

In [1]:
from huggingface_hub import notebook_login
notebook_login()

# Carregar dataset JSON

In [2]:
import json
import pandas as pd
from datasets import Dataset

data = pd.read_parquet("dataset_finetuning.parquet")

dataset = Dataset.from_pandas(data)
dataset

Dataset({
    features: ['text'],
    num_rows: 665
})

# Formatando dataset para treinamento

In [3]:
import json
from transformers import AutoTokenizer

model_name = "meta-llama/Llama-3.1-8B-Instruct"
# Load tokenizer early to format the dataset with Llama 3 Chat Template
tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

def format_example(example):
    instruction = "Convert project description into structured GitHub issues JSON."
    
    # If the dataset is already using Alpaca text, parse it back
    if 'text' in example and '### Instruction:' in example['text']:
        text = example['text']
        input_part = text.split("### Input:\n")[1].split("### Response:\n")[0].strip() if "### Input:\n" in text else ""
        response_part = text.split("### Response:\n")[1].strip() if "### Response:\n" in text else ""
        
        messages = [
            {"role": "system", "content": instruction},
            {"role": "user", "content": input_part},
            {"role": "assistant", "content": response_part}
        ]
    # If the dataset has raw input/output columns
    else:
        input_text = example.get("input", "")
        output_text = example.get("output", "")
        if isinstance(output_text, dict) or isinstance(output_text, list):
            output_text = json.dumps(output_text, indent=2)
            
        messages = [
            {"role": "system", "content": instruction},
            {"role": "user", "content": input_text},
            {"role": "assistant", "content": output_text}
        ]
        
    formatted = tokenizer.apply_chat_template(messages, tokenize=False)
    return {"text": formatted}

# Apply formatting
dataset = dataset.map(format_example)
if "input" in dataset.column_names:
    dataset = dataset.remove_columns(["input", "output"])
dataset


Map:   0%|          | 0/665 [00:00<?, ? examples/s]

Dataset({
    features: ['text'],
    num_rows: 665
})

# Carregar modelo com QLoRA (4-bit)

In [4]:
import torch
# Para funcionar em GPU nvidia, python 3.11.9 exatamente
#ver se a GPU esta funcionando (nvidia-smi)- para ver a GPU cuda cors version
# py -3.11 -m install torch torchvision --index-url https://download.pytorch.org/whl/cu130 (para instalar com compatibilidade aos cudas da nvidia)
# py -3.11 -m pip uninstall torch torchvision torchaudio (para se der errado)
# https://pytorch.org/get-started/locally/
print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0))

True
NVIDIA GeForce RTX 5070


In [5]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

model_name = "meta-llama/Llama-3.1-8B-Instruct"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True
)

# Tokenizer already loaded and configured above

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    device_map="cuda",
    dtype=torch.float16
)


W0328 16:04:10.012000 6044 site-packages\torch\distributed\elastic\multiprocessing\redirects.py:29] NOTE: Redirects are currently not supported in Windows or MacOs.


Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

# Configurar LoRA

In [6]:
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules="all-linear", #original ["q_proj", "v_proj"]
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

model = prepare_model_for_kbit_training(model) #adicionado para corrigir pesos

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

trainable params: 41,943,040 || all params: 8,072,204,288 || trainable%: 0.5196


# Tokenização

In [7]:
def tokenize(example):
    return tokenizer(
        example["text"],
        truncation=True,
        max_length=8192
    )

tokenized_dataset = dataset.map(tokenize, batched=True)

# Verificar quantos exemplos foram truncados
truncated = sum(1 for ids in tokenized_dataset["input_ids"] if len(ids) == 8192)
total = len(tokenized_dataset)
print(f"Exemplos truncados (atingiram 8192 tokens): {truncated}/{total} ({100*truncated/total:.1f}%)")
print(f"Se for alto (>10%), considere filtrar esses exemplos para evitar respostas JSON incompletas no treino.")

Map:   0%|          | 0/665 [00:00<?, ? examples/s]

Exemplos truncados (atingiram 8192 tokens): 0/665 (0.0%)
Se for alto (>10%), considere filtrar esses exemplos para evitar respostas JSON incompletas no treino.


# Treinamento

In [8]:
from trl import SFTTrainer, SFTConfig

training_args = SFTConfig(
    output_dir="./llama-backlog",
    per_device_train_batch_size=1,
    gradient_accumulation_steps=4,
    num_train_epochs=3,
    logging_steps=10,
    save_steps=100,
    learning_rate=2e-4,
    fp16=False,
    bf16=True,
    optim="paged_adamw_8bit",
    report_to="none",
    lr_scheduler_type="cosine",
    warmup_steps=15,
    dataset_text_field="text",
    packing=False,
)

trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=dataset,
    processing_class=tokenizer,
)

trainer.train()


Adding EOS to train dataset:   0%|          | 0/665 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/665 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/665 [00:00<?, ? examples/s]

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 128009, 'pad_token_id': 128009}.


Step,Training Loss
10,1.343342
20,0.909043
30,0.759423
40,0.695286
50,0.660824
60,0.608309
70,0.580064
80,0.576132
90,0.568765
100,0.577290


TrainOutput(global_step=501, training_loss=0.4472978694174818, metrics={'train_runtime': 3677.8667, 'train_samples_per_second': 0.542, 'train_steps_per_second': 0.136, 'total_flos': 9.244878231608525e+16, 'train_loss': 0.4472978694174818, 'entropy': 0.25989410281181335, 'num_tokens': 2041659.0, 'mean_token_accuracy': 0.9266862273216248, 'epoch': 3.0})

# Salvar modelo fine-tuned

In [9]:
model.save_pretrained("llama-backlog-lora")
tokenizer.save_pretrained("llama-backlog-lora")

('llama-backlog-lora\\tokenizer_config.json',
 'llama-backlog-lora\\chat_template.jinja',
 'llama-backlog-lora\\tokenizer.json')

# Carregar o Modelo (Apenas se não foi treinado na hora)

In [1]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import PeftModel
# 1. Configurações e modelo base
base_model_name = "meta-llama/Llama-3.1-8B-Instruct"
adapter_model_path = "llama-backlog-lora"
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True
)
base_model = AutoModelForCausalLM.from_pretrained(
    base_model_name,
    quantization_config=bnb_config,
    device_map="cuda",
    dtype=torch.float16
)
tokenizer = AutoTokenizer.from_pretrained(base_model_name)
tokenizer.pad_token = tokenizer.eos_token
# 2. CARREGAR OS PESOS GERADOS NO SEU FINETUNING (LORA)
model = PeftModel.from_pretrained(base_model, adapter_model_path)
# 3. Colocar o modelo em modo de avaliação/inferência
model.eval()


W0328 19:06:06.876000 13704 site-packages\torch\distributed\elastic\multiprocessing\redirects.py:29] NOTE: Redirects are currently not supported in Windows or MacOs.


Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

PeftModelForCausalLM(
  (base_model): LoraModel(
    (model): LlamaForCausalLM(
      (model): LlamaModel(
        (embed_tokens): Embedding(128256, 4096)
        (layers): ModuleList(
          (0-31): 32 x LlamaDecoderLayer(
            (self_attn): LlamaAttention(
              (q_proj): lora.Linear4bit(
                (base_layer): Linear4bit(in_features=4096, out_features=4096, bias=False)
                (lora_dropout): ModuleDict(
                  (default): Dropout(p=0.05, inplace=False)
                )
                (lora_A): ModuleDict(
                  (default): Linear(in_features=4096, out_features=16, bias=False)
                )
                (lora_B): ModuleDict(
                  (default): Linear(in_features=16, out_features=4096, bias=False)
                )
                (lora_embedding_A): ParameterDict()
                (lora_embedding_B): ParameterDict()
                (lora_magnitude_vector): ModuleDict()
              )
              (k_proj): lor

# Testar o modelo

In [2]:
from transformers import TextStreamer

messages = [
    {"role": "system", "content": "Convert project description into structured GitHub issues JSON."},
    {"role": "user", "content": """Criar uma plataforma inteligente que utilize IA para processar documentação (como PDFs) e automatizar a conversão de requisitos técnicos e de negócio em issues estruturadas no GitHub. O sistema deve transformar documentos brutos em tarefas organizadas, com títulos claros, descrições técnicas detalhadas, critérios de aceitação e categorização automática (Feature, Bug, Improvement, Technical Task).

A plataforma deve:

* Automatizar a ingestão de dados, convertendo PDFs em issues completas.
* Padronizar a organização do backlog com aplicação automática de labels e segmentações.
* Gerar estimativas de tempo total de produção e sugerir divisão em sprints com base na densidade do backlog.
* Extrair e documentar requisitos funcionais, não-funcionais, regras de negócio e restrições técnicas.
* Identificar inconsistências entre documentos e sugerir melhorias arquiteturais por meio de relatórios executivos sob demanda.
* Reduzir o gap de comunicação entre negócio e desenvolvimento, garantindo que as tarefas reflitam fielmente os objetivos estratégicos da empresa."""}
]

# Cria o efeito de "maquininha de escrever" para vermos ao vivo
streamer = TextStreamer(tokenizer, skip_prompt=True)

prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)

inputs = tokenizer(prompt, return_tensors="pt").to("cuda")

output = model.generate(
    **inputs,
    do_sample=True,
    temperature=0.4,
    top_p=0.9,
    repetition_penalty=1.2,
    max_new_tokens=5000,
    streamer=streamer,
    eos_token_id=[tokenizer.eos_token_id, tokenizer.convert_tokens_to_ids("<|eot_id|>")],
    pad_token_id=tokenizer.eos_token_id
    )

print(tokenizer.decode(output[0][inputs['input_ids'].shape[-1]:], skip_special_tokens=True))

[{"title": "feat(api): POST /documents - upload PDF and trigger conversion", "body": "### Objetivo\nPermitir que usuários enviem documentos PDF que serão automaticamente convertidos em issues GitHub.\n\n### Descrição\nImplementar endpoint RESTful `POST /documents` que recebe multipart/form-data contendo o arquivo PDF. Criar DTO `DocumentUploadDto` com validação de tipo e tamanho. Utilizar serviço `PdfConversionService` para chamar um motor NLP/OCR externo via HTTP, gerando um payload JSON compatível com a API GitHub Issues. Persistir o status inicial (`queued`) no banco PostgreSQL usando TypeORM repository. Retornar resposta `{ id, status }`. Tecnologias: NestJS, TypeScript, class-validator, Axios.\n\n### Critérios de aceitação\n- [ ] Endpoint responde 200 com corpo `{ \"id\": <uuid>, \"status\": \"queued\" }`\n- [ ] Validação rejeita arquivos não-PDF retornando 400\n- [ ] Payload interno segue schema definido pelo GitHub Issue creation spec\n- [ ] Registro salvo na tabela `document_qu

In [ ]:
messages = [
    {"role": "system", "content": "Convert project description into structured GitHub issues JSON."},
    {"role": "user", "content": """Desenvolver uma plataforma web de gerenciamento de eventos e apostas em resultados futuros, onde usuários possam criar eventos, apostar em resultados e acompanhar estatísticas em tempo real. O sistema deve permitir cadastro de usuários, criação e moderação de eventos, registro de apostas, controle financeiro e geração de relatórios analíticos.

A aplicação deverá possuir backend com API REST, interface web responsiva e um sistema de moderação para validar conteúdos antes de serem publicados.

O projeto deve contemplar diferentes tipos de issues (Feature, Bug, Improvement, Technical Task) ao transformar os requisitos abaixo em tarefas estruturadas.

A plataforma deve:

Permitir cadastro, login e autenticação de usuários, com recuperação de senha e validação de email.

Permitir que usuários criem eventos futuros (ex.: jogos, eleições, lançamentos de produtos) nos quais outras pessoas possam apostar.

Implementar sistema de apostas com duas ou mais opções de resultado, registrando valor apostado e histórico de apostas do usuário.

Incluir painel de moderação, onde administradores possam aprovar ou rejeitar eventos criados pelos usuários.

Gerenciar carteira virtual, permitindo depósitos, retiradas e visualização de saldo.

Disponibilizar dashboard com estatísticas, como volume total apostado, eventos mais populares e desempenho do usuário.

Garantir segurança e integridade dos dados, incluindo criptografia de senhas, proteção contra ataques comuns e validação de entradas.

Implementar notificações por email ou sistema interno para informar usuários sobre aprovação de eventos, resultados e ganhos.

Além das funcionalidades principais, o sistema deve:

Identificar possíveis bugs ou inconsistências funcionais, como falhas na validação de apostas ou cálculo incorreto de ganhos.

Propor melhorias de experiência do usuário, como filtros de eventos, busca avançada e interface mais intuitiva.

Incluir tarefas técnicas, como configuração de banco de dados, integração com serviços de pagamento, criação de testes automatizados e pipelines de CI/CD.

O resultado esperado é a geração automática de um backlog estruturado em issues do GitHub, contendo:

Título claro da tarefa

Descrição técnica detalhada

Tipo de issue (Feature, Bug, Improvement ou Technical Task)

Critérios de aceitação

Sugestão de prioridade

Estimativa de esforço

Labels apropriadas para organização do projeto."""}
]

prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)

inputs = tokenizer(prompt, return_tensors="pt").to("cuda")

output = model.generate(
    **inputs,
    do_sample=True,
    temperature=0.4,
    top_p=0.9,
    repetition_penalty=1.2,
    max_new_tokens=5000,
    eos_token_id=[tokenizer.eos_token_id, tokenizer.convert_tokens_to_ids("<|eot_id|>")],
    pad_token_id=tokenizer.eos_token_id)

print(tokenizer.decode(output[0][inputs['input_ids'].shape[-1]:], skip_special_tokens=True))

# Ver total de parâmetros

In [3]:
total_params = sum(p.numel() for p in model.parameters())
print(f"Total de parâmetros: {total_params:,}")

Total de parâmetros: 4,582,543,360
